# Random Forest — Flight Arrival Delay Prediction

Trains a Random Forest classifier (PySpark MLlib) on US domestic flight data (2018–2022) to predict arrival delays ≥ 15 minutes. Evaluates on validation (2023) and test (2024) sets.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Install PySpark

In [3]:
!pip install pyspark --quiet

### Set Java Home and Verify PySpark

In [4]:
import os
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import precision_recall_curve, auc


### Start Spark Session

In [5]:
spark = (
    SparkSession.builder
    .appName("RandomForestModeling")
    .master("local[*]")
    .config("spark.driver.memory", "35g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")
spark.sparkContext.setLogLevel("WARN")
spark

## Load Data

### Check and Download Data from Zenodo

Downloads `train.parquet`, `val.parquet`, `test.parquet` from Zenodo if not already on Drive.

In [ ]:
BASE_PATH  = '/content/drive/MyDrive/flight_data'
ZENODO_URL = 'https://zenodo.org/records/20489802/files'

os.makedirs(BASE_PATH, exist_ok=True)

for filename in ['train.parquet', 'val.parquet', 'test.parquet']:
    local_path = f'{BASE_PATH}/{filename}'
    if not os.path.exists(local_path):
        print(f'Downloading {filename}...')
        urllib.request.urlretrieve(f'{ZENODO_URL}/{filename}?download=1', local_path)
        print(f'  Done: {local_path}')
    else:
        print(f'Found: {local_path}')


### Load Train and Validation Data

In [ ]:
TRAIN_PATH    = f'{BASE_PATH}/train.parquet'
VALIDATE_PATH = f'{BASE_PATH}/val.parquet'
TEST_PATH     = f'{BASE_PATH}/test.parquet'

train_df    = spark.read.parquet(TRAIN_PATH)
validate_df = spark.read.parquet(VALIDATE_PATH)

print('Data loaded.')


### Inspect Columns

In [7]:
print(train_df.columns)

['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Origin', 'Dest', 'CRSDepTime', 'DepTimeBlk', 'CRSArrTime', 'ArrDel15', 'CRSElapsedTime', 'Distance', 'DistanceGroup', 'date', 'dep_hour', 'arr_hour', 'dep_hour_minus2', 'arr_hour_minus2', 'origin_temp_f', 'origin_dewpoint_f', 'origin_humidity', 'origin_feels_like_f', 'origin_wind_kts', 'origin_gust_kts', 'origin_visibility', 'origin_precip_in', 'origin_wx_codes', 'origin_is_rain', 'origin_is_snow', 'origin_is_fog', 'origin_low_visibility', 'origin_high_wind', 'origin_severe_weather', 'dest_temp_f', 'dest_dewpoint_f', 'dest_humidity', 'dest_feels_like_f', 'dest_wind_kts', 'dest_gust_kts', 'dest_visibility', 'dest_precip_in', 'dest_wx_codes', 'dest_is_rain', 'dest_is_snow', 'dest_is_fog', 'dest_low_visibility', 'dest_high_wind', 'dest_severe_weather', 'is_weekend', 'is_holiday', 'origin_weather_missing', 'dest_weather_missing', 'carrier_delay_rate_30d', 'carrier_

## Feature Engineering

### Cast Label Column to Double

Spark ML requires the label column to be of type Double.

In [8]:
# Cast ArrDel15 to Double — Spark ML requires numeric label column
train_df    = train_df.withColumn("ArrDel15", F.col("ArrDel15").cast(DoubleType()))
validate_df = validate_df.withColumn("ArrDel15", F.col("ArrDel15").cast(DoubleType()))

print("ArrDel15 dtype:", dict(train_df.dtypes)["ArrDel15"])

### Define Feature Columns

Excludes target, leakage-prone, and identifier columns.

In [9]:
feature_cols = [
    "Month", "DayofMonth", "DayOfWeek", "dep_hour", "arr_hour",
    "CRSElapsedTime", "Distance", "DistanceGroup",
    "is_weekend", "is_holiday",
    "origin_delay_rate", "dest_delay_rate",
    "carrier_delay_rate_30d", "carrier_delay_rate_90d",
    "origin_delay_rate_30d", "origin_delay_rate_90d",
    "dest_delay_rate_30d", "dest_delay_rate_90d",
    "origin_departures_3h",
    "origin_temp_f", "origin_dewpoint_f", "origin_humidity",
    "origin_feels_like_f", "origin_wind_kts", "origin_gust_kts",
    "origin_visibility", "origin_precip_in",
    "origin_is_rain", "origin_is_snow", "origin_is_fog",
    "origin_low_visibility", "origin_high_wind", "origin_severe_weather",
    "dest_temp_f", "dest_dewpoint_f", "dest_humidity",
    "dest_feels_like_f", "dest_wind_kts", "dest_gust_kts",
    "dest_visibility", "dest_precip_in",
    "dest_is_rain", "dest_is_snow", "dest_is_fog",
    "dest_low_visibility", "dest_high_wind", "dest_severe_weather",
]

print(f"Total features: {len(feature_cols)}")

Total features: 47


### Stratified Sampling and Feature Assembly

Samples 20% of training data with stratification to preserve class balance, then assembles features into a vector.

In [10]:
SAMPLE_FRACTION = 0.20

# Stratified sample by Year x Month x Class — preserves temporal and class distribution
df_train_keyed = train_df.withColumn(
    'strat_key',
    F.concat_ws('_',
        F.col('Year').cast('string'),
        F.col('Month').cast('string'),
        F.col('ArrDel15').cast('string')
    )
)

strata = [row['strat_key'] for row in
          df_train_keyed.select('strat_key').distinct().collect()]
fractions = {s: SAMPLE_FRACTION for s in strata}

train_sample = (
    df_train_keyed
    .sampleBy('strat_key', fractions=fractions, seed=42)
    .drop('strat_key')
)

train_n = train_sample.count()
print(f"Stratified sample : {train_n:,} rows ({100*SAMPLE_FRACTION:.0f}% of train)")

# Verify class balance preserved
counts = train_sample.groupBy('ArrDel15').count().collect()
counts_dict = {row['ArrDel15']: row['count'] for row in counts}
sample_rate = counts_dict[1] / (counts_dict[0] + counts_dict[1])
actual_ratio = counts_dict[0] / counts_dict[1]
print(f"Sample delay rate : {sample_rate:.4f}  (original: 0.1785)")
print(f"Actual class ratio: {actual_ratio:.2f}  (using scale_pos_weight: 3.5)")

# Year distribution check
print("\nSampled train by year:")
train_sample.groupBy('Year').count().orderBy('Year').show()

# Weight column — using 3.5 instead of full ratio (4.59) to balance precision/recall
scale_pos_weight = 3.5
train_sample = train_sample.withColumn(
    'weight',
    F.when(F.col('ArrDel15') == 1, scale_pos_weight).otherwise(1.0)
)

# Drop features column if already exists from a previous run
if 'features' in train_sample.columns:
    train_sample = train_sample.drop('features')
if 'features' in validate_df.columns:
    validate_df = validate_df.drop('features')

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

train_sample = assembler.transform(train_sample)
validate_df  = assembler.transform(validate_df)

print("Assembler applied.")

## Model Training

### Train Random Forest

100 trees, max depth 12, trained on the assembled feature vector.

In [11]:
rf = RandomForestClassifier(
    labelCol="ArrDel15",
    featuresCol="features",
    weightCol="weight",
    numTrees=100,
    maxDepth=12,
    maxBins=64,
    featureSubsetStrategy="onethird",
    minInstancesPerNode=5,
    seed=42
)

print("Training Random Forest...")
rf_model = rf.fit(train_sample)
print("Training complete.")

## Evaluation

### Validation Metrics (2023)

In [12]:
predictions = rf_model.transform(validate_df)

roc_auc = BinaryClassificationEvaluator(
    labelCol="ArrDel15", metricName="areaUnderROC"
).evaluate(predictions)

pr_auc = BinaryClassificationEvaluator(
    labelCol="ArrDel15", metricName="areaUnderPR"
).evaluate(predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="ArrDel15", metricName="f1"
).evaluate(predictions)

print(f"Validation ROC-AUC : {roc_auc:.4f}")
print(f"Validation PR-AUC  : {pr_auc:.4f}")
print(f"Validation F1      : {f1:.4f}")

# Class-specific precision and recall from confusion matrix
cm_rows = predictions.groupBy('ArrDel15', 'prediction').count().collect()
cm = {(int(row['ArrDel15']), int(row['prediction'])): row['count'] for row in cm_rows}

tn = cm.get((0, 0), 0)
fp = cm.get((0, 1), 0)
fn = cm.get((1, 0), 0)
tp = cm.get((1, 1), 0)

delayed_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
delayed_recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
ontime_precision  = tn / (tn + fn) if (tn + fn) > 0 else 0
ontime_recall     = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"\n{'Class':<12} {'Precision':>10} {'Recall':>10}")
print(f"{'-'*34}")
print(f"{'On-Time':<12} {ontime_precision:>10.4f} {ontime_recall:>10.4f}")
print(f"{'Delayed':<12} {delayed_precision:>10.4f} {delayed_recall:>10.4f}")

# Raw confusion matrix counts
print("\nConfusion matrix (counts):")
predictions.groupBy('ArrDel15', 'prediction').count().orderBy('ArrDel15', 'prediction').show()

### Train vs Validation Comparison

In [13]:
train_preds = rf_model.transform(train_sample)

train_roc = BinaryClassificationEvaluator(labelCol="ArrDel15", metricName="areaUnderROC").evaluate(train_preds)
train_pr  = BinaryClassificationEvaluator(labelCol="ArrDel15", metricName="areaUnderPR").evaluate(train_preds)
train_f1  = MulticlassClassificationEvaluator(labelCol="ArrDel15", metricName="f1").evaluate(train_preds)

cm_rows = train_preds.groupBy('ArrDel15', 'prediction').count().collect()
cm_t = {(int(row['ArrDel15']), int(row['prediction'])): row['count'] for row in cm_rows}
tn_t, fp_t, fn_t, tp_t = cm_t.get((0,0),0), cm_t.get((0,1),0), cm_t.get((1,0),0), cm_t.get((1,1),0)

train_delayed_precision = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
train_delayed_recall    = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0

print(f"{'Metric':<22} {'Train':>8}  {'Val':>8}  {'Gap':>8}")
print("-" * 52)
print(f"{'ROC-AUC':<22} {train_roc:>8.4f}  {roc_auc:>8.4f}  {train_roc - roc_auc:>8.4f}")
print(f"{'PR-AUC':<22} {train_pr:>8.4f}  {pr_auc:>8.4f}  {train_pr - pr_auc:>8.4f}")
print(f"{'F1':<22} {train_f1:>8.4f}  {f1:>8.4f}  {train_f1 - f1:>8.4f}")
print(f"{'Delayed Precision':<22} {train_delayed_precision:>8.4f}  {delayed_precision:>8.4f}  {train_delayed_precision - delayed_precision:>8.4f}")
print(f"{'Delayed Recall':<22} {train_delayed_recall:>8.4f}  {delayed_recall:>8.4f}  {train_delayed_recall - delayed_recall:>8.4f}")

### Confusion Matrix — Validation Set (Default Threshold = 0.50)

In [14]:
cm_pd = (
    predictions.groupBy('ArrDel15', 'prediction').count()
    .toPandas()
    .pivot(index='ArrDel15', columns='prediction', values='count')
    .fillna(0).astype(int)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm_pd, annot=True, fmt=',d', cmap='Blues', ax=axes[0],
            xticklabels=['On-Time', 'Delayed'],
            yticklabels=['On-Time', 'Delayed'])
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Row-normalized (recall view)
cm_pct = cm_pd.div(cm_pd.sum(axis=1), axis=0) * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
            xticklabels=['On-Time', 'Delayed'],
            yticklabels=['On-Time', 'Delayed'],
            cbar_kws={'label': '%'})
axes[1].set_title('Confusion Matrix (Row %)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

### Feature Importances

Top 20 features ranked by Gini importance.

In [15]:
# Feature importances — top 20

importances = rf_model.featureImportances
feat_imp_df = pd.DataFrame({
    "feature":    feature_cols,
    "importance": importances.toArray()
}).sort_values("importance", ascending=False)

print(feat_imp_df.head(20).to_string(index=False))

In [16]:
top20 = feat_imp_df.head(20).sort_values("importance")

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top20["feature"], top20["importance"], color="steelblue", alpha=0.85)
ax.set_title("Top 20 Feature Importances — Random Forest", fontsize=13, fontweight="bold")
ax.set_xlabel("Importance (Gini)", fontsize=11)
ax.set_ylabel("Feature", fontsize=11)
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("rf_feature_importances.png", dpi=150, bbox_inches="tight")
plt.show()

### Precision-Recall Curve — Validation Set (2023)

In [17]:
# Collect full validation set scores
pr_data = (
    predictions
    .withColumn('prob_array', vector_to_array(F.col('probability')))
    .select(F.col('ArrDel15').alias('label'), F.col('prob_array')[1].alias('score'))
    .toPandas()
)

print(f"Validation size: {len(pr_data):,} rows")
print(f"Validation delay rate: {pr_data['label'].mean():.4f}")

# Compute PR curve
precision_vals, recall_vals, thresholds = precision_recall_curve(
    pr_data['label'], pr_data['score']
)
pr_auc_score = auc(recall_vals, precision_vals)

# Find the point closest to threshold 0.30
idx_030 = (abs(thresholds - 0.30)).argmin()
p_030   = precision_vals[idx_030]
r_030   = recall_vals[idx_030]

# Baseline (random classifier = delay rate)
baseline = pr_data['label'].mean()

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(recall_vals, precision_vals, color='steelblue', linewidth=2,
        label=f'Random Forest (PR-AUC = {pr_auc_score:.4f})')
ax.axhline(baseline, color='gray', linestyle='--', linewidth=1.2,
           label=f'Baseline (random classifier = {baseline:.2f})')

# Mark operating point at threshold 0.30
ax.scatter(r_030, p_030, color='red', zorder=5, s=100,
           label=f'Threshold 0.30  (P={p_030:.3f}, R={r_030:.3f})')
ax.annotate(f'T=0.30\nP={p_030:.3f}, R={r_030:.3f}',
            xy=(r_030, p_030), xytext=(r_030 - 0.18, p_030 + 0.04),
            fontsize=9, color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.2))

ax.set_title('Precision-Recall Curve — Random Forest, Validation Set 2023',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(fontsize=10, loc='upper right')
ax.grid(alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('rf_pr_curve_val.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPR-AUC (full val): {pr_auc_score:.4f}")
print(f"At threshold 0.30: Precision={p_030:.4f}, Recall={r_030:.4f}")

### Threshold Tuning

Sweeps thresholds to find the optimal F1 score on the validation set.

In [18]:
# Convert probability vector to array once for efficiency
predictions_arr = predictions.withColumn('prob_array', vector_to_array(F.col('probability')))

print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Flagged':>10}")
print("-" * 55)

for threshold in [0.5, 0.4, 0.35, 0.3, 0.25, 0.2]:
    preds = predictions_arr.withColumn(
        'prediction', (F.col('prob_array')[1] >= threshold).cast('double')
    )
    cm_rows = preds.groupBy('ArrDel15', 'prediction').count().collect()
    cm = {(int(r['ArrDel15']), int(r['prediction'])): r['count'] for r in cm_rows}
    tp = cm.get((1,1), 0); fp = cm.get((0,1), 0); fn = cm.get((1,0), 0)
    prec  = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec   = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_sc = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    flagged = tp + fp
    print(f"{threshold:>10.2f} {prec:>10.4f} {rec:>10.4f} {f1_sc:>10.4f} {flagged:>10,}")

### Test Set Evaluation (2024) at Threshold 0.30

In [19]:
# Load and preprocess test set
test_df = spark.read.parquet(TEST_PATH)
test_df = test_df.withColumn("ArrDel15", F.col("ArrDel15").cast(DoubleType()))

if 'features' in test_df.columns:
    test_df = test_df.drop('features')

test_df = assembler.transform(test_df)

# Generate predictions
test_predictions = rf_model.transform(test_df)

# ── Threshold-independent metrics ──────────────────────────────────────────────
test_roc_auc = BinaryClassificationEvaluator(
    labelCol="ArrDel15", metricName="areaUnderROC"
).evaluate(test_predictions)

test_pr_auc = BinaryClassificationEvaluator(
    labelCol="ArrDel15", metricName="areaUnderPR"
).evaluate(test_predictions)

# ── Apply threshold 0.30 ───────────────────────────────────────────────────────
THRESHOLD = 0.30

test_preds_thresh = (
    test_predictions
    .withColumn('prob_array', vector_to_array(F.col('probability')))
    .withColumn('prediction', (F.col('prob_array')[1] >= THRESHOLD).cast('double'))
)

# Confusion matrix
cm_rows = test_preds_thresh.groupBy('ArrDel15', 'prediction').count().collect()
cm = {(int(r['ArrDel15']), int(r['prediction'])): r['count'] for r in cm_rows}
tn = cm.get((0, 0), 0)
fp = cm.get((0, 1), 0)
fn = cm.get((1, 0), 0)
tp = cm.get((1, 1), 0)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

# ── Print results ──────────────────────────────────────────────────────────────
print(f"Test Set Evaluation (2024) — Threshold = {THRESHOLD}\n")
print(f"  ROC-AUC   : {test_roc_auc:.4f}")
print(f"  PR-AUC    : {test_pr_auc:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1        : {f1:.4f}")

print(f"\nConfusion Matrix (counts):")
print(f"                  Predicted On-Time   Predicted Delayed")
print(f"  Actual On-Time  {tn:>18,}   {fp:>17,}")
print(f"  Actual Delayed  {fn:>18,}   {tp:>17,}")

total_actual_delayed = tp + fn
total_flagged        = tp + fp
print(f"\n  Actual delayed flights : {total_actual_delayed:,}")
print(f"  Flagged as delayed     : {total_flagged:,}")
print(f"  Delays caught (TP)     : {tp:,}  ({100*recall:.1f}% of actual delays)")
print(f"  False alarms (FP)      : {fp:,}  ({100*fp/total_flagged:.1f}% of alerts sent)")

### Confusion Matrix — Test Set (Threshold = 0.30)

In [20]:
# Confusion matrix — Test Set 2024, Threshold 0.30
# tn, fp, fn, tp are already computed in the cell above
cm = np.array([[tn, fp], [fn, tp]])

cm_pct = cm.astype(float)
cm_pct[0] = cm[0] / cm[0].sum() * 100
cm_pct[1] = cm[1] / cm[1].sum() * 100

labels_count = [[f'{tn:,}', f'{fp:,}'], [f'{fn:,}', f'{tp:,}']]
labels_pct   = [[f'{cm_pct[0,0]:.1f}%', f'{cm_pct[0,1]:.1f}%'],
                [f'{cm_pct[1,0]:.1f}%', f'{cm_pct[1,1]:.1f}%']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=labels_count, fmt='', cmap='Blues',
            xticklabels=['On-Time', 'Delayed'],
            yticklabels=['On-Time', 'Delayed'],
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('Actual', fontsize=11)

sns.heatmap(cm_pct, annot=labels_pct, fmt='', cmap='Blues',
            xticklabels=['On-Time', 'Delayed'],
            yticklabels=['On-Time', 'Delayed'],
            vmin=0, vmax=100,
            cbar_kws={'label': '%'},
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Confusion Matrix (Row %)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted', fontsize=11)
axes[1].set_ylabel('Actual', fontsize=11)

fig.suptitle(
    f'Random Forest — Test Set 2024, Threshold = 0.30  (Recall = {recall*100:.1f}%)',
    fontsize=12, y=1.02
)

plt.tight_layout()
plt.savefig('rf_confusion_matrix_test.png', dpi=150, bbox_inches='tight')
plt.show()

### Monthly Stability Analysis — 2023 Validation vs 2024 Test

Checks whether model performance is stable across months.

In [21]:
THRESHOLD = 0.30
months = list(range(1, 13))
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# ── Validation monthly F1 at threshold 0.30 ───────────────────────────────────
val_preds_thresh = (
    predictions
    .withColumn('prob_array', vector_to_array(F.col('probability')))
    .withColumn('prediction', (F.col('prob_array')[1] >= THRESHOLD).cast('double'))
)

val_monthly_cm = (
    val_preds_thresh
    .groupBy('Month')
    .agg(
        F.sum(((F.col('prediction') == 1) & (F.col('ArrDel15') == 1)).cast('int')).alias('tp'),
        F.sum(((F.col('prediction') == 1) & (F.col('ArrDel15') == 0)).cast('int')).alias('fp'),
        F.sum(((F.col('prediction') == 0) & (F.col('ArrDel15') == 1)).cast('int')).alias('fn'),
    )
    .orderBy('Month')
    .toPandas()
)
val_monthly_cm['precision'] = val_monthly_cm['tp'] / (val_monthly_cm['tp'] + val_monthly_cm['fp']).replace(0, np.nan)
val_monthly_cm['recall']    = val_monthly_cm['tp'] / (val_monthly_cm['tp'] + val_monthly_cm['fn']).replace(0, np.nan)
val_monthly_cm['f1']        = (2 * val_monthly_cm['precision'] * val_monthly_cm['recall'] /
                               (val_monthly_cm['precision'] + val_monthly_cm['recall']).replace(0, np.nan))
val_f1 = val_monthly_cm['f1'].values

# ── Test monthly metrics at threshold 0.30 ────────────────────────────────────
test_preds_thresh.cache()

monthly_cm = (
    test_preds_thresh
    .groupBy('Month')
    .agg(
        F.sum(((F.col('prediction') == 1) & (F.col('ArrDel15') == 1)).cast('int')).alias('tp'),
        F.sum(((F.col('prediction') == 1) & (F.col('ArrDel15') == 0)).cast('int')).alias('fp'),
        F.sum(((F.col('prediction') == 0) & (F.col('ArrDel15') == 1)).cast('int')).alias('fn'),
        F.sum(((F.col('prediction') == 0) & (F.col('ArrDel15') == 0)).cast('int')).alias('tn'),
        F.count('*').alias('n'),
        F.sum(F.col('ArrDel15')).alias('positives'),
    )
    .orderBy('Month')
    .toPandas()
)

monthly_cm['base_rate'] = monthly_cm['positives'] / monthly_cm['n']
monthly_cm['precision'] = monthly_cm['tp'] / (monthly_cm['tp'] + monthly_cm['fp']).replace(0, np.nan)
monthly_cm['recall']    = monthly_cm['tp'] / (monthly_cm['tp'] + monthly_cm['fn']).replace(0, np.nan)
monthly_cm['f1']        = (2 * monthly_cm['precision'] * monthly_cm['recall'] /
                           (monthly_cm['precision'] + monthly_cm['recall']).replace(0, np.nan))

# ── ROC-AUC per month (test) ───────────────────────────────────────────────────
roc_evaluator = BinaryClassificationEvaluator(labelCol="ArrDel15", metricName="areaUnderROC")
roc_aucs = []
for m in months:
    month_df = test_preds_thresh.filter(F.col('Month') == m)
    roc_aucs.append(roc_evaluator.evaluate(month_df))
monthly_cm['roc_auc']    = roc_aucs
monthly_cm['Month_name'] = month_names
test_f1 = monthly_cm['f1'].values

# ── Print summary table (test) ─────────────────────────────────────────────────
display_cols = ['Month_name', 'base_rate', 'precision', 'recall', 'f1', 'roc_auc']
display_df = monthly_cm[display_cols].copy()
display_df.columns = ['Month', 'Base Rate', 'Precision', 'Recall', 'F1', 'ROC-AUC']

print("Monthly Stability Analysis — Test Set (2024) at Threshold 0.30\n")
print(display_df.to_string(index=False, float_format='%.4f'))
print("-" * 62)
for stat, fn_stat in [('Mean', 'mean'), ('Std', 'std')]:
    row = getattr(display_df[['Base Rate', 'Precision', 'Recall', 'F1', 'ROC-AUC']], fn_stat)()
    print(f"{'':6} {stat:<10} {row['Base Rate']:>9.4f} {row['Precision']:>10.4f} "
          f"{row['Recall']:>7.4f} {row['F1']:>7.4f} {row['ROC-AUC']:>9.4f}")

# Stability scores
val_mean  = np.mean(val_f1);  val_std  = np.std(val_f1, ddof=1)
test_mean = np.mean(test_f1); test_std = np.std(test_f1, ddof=1)
print(f"\nStability Score — Val F1  : mean={val_mean:.4f}, std={val_std:.4f}")
print(f"Stability Score — Test F1 : mean={test_mean:.4f}, std={test_std:.4f}")

# ── Plot: Val vs Test F1 per month ────────────────────────────────────────────
bar_width = 0.4
x = np.arange(len(month_names))

fig, ax = plt.subplots(figsize=(14, 6))
bars_val  = ax.bar(x - bar_width/2, val_f1,  bar_width, color='#4A90E2', alpha=0.85, label='2023 Val')
bars_test = ax.bar(x + bar_width/2, test_f1, bar_width, color='#F18F35', alpha=0.85, label='2024 Test')

ax.axhline(val_mean,  color='#4A90E2', linestyle='--', linewidth=1.5,
           label=f'Val mean  ({val_mean:.3f})')
ax.axhline(test_mean, color='#F18F35', linestyle='--', linewidth=1.5,
           label=f'Test mean ({test_mean:.3f})')

for bar in bars_val:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8, color='#1F4E8B')
for bar in bars_test:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8, color='#B85B1F')

ax.set_title('Monthly F1 — Random Forest, 2023 Validation vs 2024 Test (Threshold 0.30)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_xlabel('Month', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(month_names)
ax.set_ylim(0, max(max(val_f1), max(test_f1)) * 1.2)
ax.legend(fontsize=10, loc='upper right')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('rf_monthly_f1_val_vs_test.png', dpi=150, bbox_inches='tight')
plt.show()